In [0]:
from pyspark.sql.functions import *

In [0]:
spark.conf.set(
    "fs.azure.account.key.<storageaccountname>.dfs.core.windows.net",
     <"storagekey">
)

In [0]:
jobs_source = "abfss://silver@projectstorage2612.dfs.core.windows.net/jobs_clean"
company_source = "abfss://silver@projectstorage2612.dfs.core.windows.net/company_clean"
salary_source = "abfss://silver@projectstorage2612.dfs.core.windows.net/salary_clean"
location_source = "abfss://silver@projectstorage2612.dfs.core.windows.net/location_clean"
experience_source = "abfss://silver@projectstorage2612.dfs.core.windows.net/experience_clean"
posting_source = "abfss://silver@projectstorage2612.dfs.core.windows.net/posting_clean"

company_dest = "abfss://gold@projectstorage2612.dfs.core.windows.net/company_analysis"
salary_dest = "abfss://gold@projectstorage2612.dfs.core.windows.net/salary_analysis"
location_dest = "abfss://gold@projectstorage2612.dfs.core.windows.net/location_analysis"
experience_dest = "abfss://gold@projectstorage2612.dfs.core.windows.net/experience_analysis"
monthly_dest = "abfss://gold@projectstorage2612.dfs.core.windows.net/monthly_analysis"
posting_dest = "abfss://gold@projectstorage2612.dfs.core.windows.net/posting_analysis"

company_parquet = "abfss://gold@projectstorage2612.dfs.core.windows.net/company_analysis_parquet"

salary_parquet = "abfss://gold@projectstorage2612.dfs.core.windows.net/salary_analysis_parquet"

location_parquet = "abfss://gold@projectstorage2612.dfs.core.windows.net/location_analysis_parquet"

experience_parquet = "abfss://gold@projectstorage2612.dfs.core.windows.net/experience_analysis_parquet"

monthly_parquet = "abfss://gold@projectstorage2612.dfs.core.windows.net/monthly_analysis_parquet"

posting_parquet = "abfss://gold@projectstorage2612.dfs.core.windows.net/posting_analysis_parquet"

In [0]:
jobs_df = spark.read.format("delta").load(jobs_source)

company_df = spark.read.format("delta").load(company_source)

salary_df = spark.read.format("delta").load(salary_source)

location_df = spark.read.format("delta").load(location_source)

experience_df = spark.read.format("delta").load(experience_source)

posting_df = spark.read.format("delta").load(posting_source)

In [0]:
company_analysis = company_df.groupBy("company_name") \
.count() \
.orderBy(desc("count"))

company_analysis.write.format("delta") \
.mode("overwrite") \
.save(company_dest)
company_analysis.write.format("parquet") \
.mode("overwrite") \
.save(company_parquet)

In [0]:
salary_analysis = salary_df.groupBy("currency") \
.agg(
    avg("normalized_salary").alias("average_salary")
)

salary_analysis.write.format("delta") \
.mode("overwrite") \
.save(salary_dest)
salary_analysis.write.format("parquet") \
.mode("overwrite") \
.save(salary_parquet)

In [0]:
location_analysis = location_df.groupBy("location") \
.count() \
.orderBy(desc("count"))

location_analysis.write.format("delta") \
.mode("overwrite") \
.save(location_dest)
location_analysis.write.format("parquet") \
.mode("overwrite") \
.save(location_parquet)

In [0]:
experience_analysis = experience_df.groupBy(
    "formatted_experience_level"
).count()

experience_analysis.write.format("delta") \
.mode("overwrite") \
.save(experience_dest)
experience_analysis.write.format("parquet") \
.mode("overwrite") \
.save(experience_parquet)

In [0]:
monthly_analysis = posting_df.withColumn(
    "Month",
    date_format(col("original_listed_time"),"yyyy-MM")
)

monthly_analysis = monthly_analysis.groupBy("Month") \
.count() \
.orderBy("Month")

monthly_analysis.write.format("delta") \
.mode("overwrite") \
.save(monthly_dest)
monthly_analysis.write.format("parquet") \
.mode("overwrite") \
.save(monthly_parquet)

In [0]:
posting_analysis = posting_df.groupBy("posting_domain") \
.count() \
.orderBy(desc("count"))

posting_analysis.write.format("delta") \
.mode("overwrite") \
.save(posting_dest)
posting_analysis.write.format("parquet") \
.mode("overwrite") \
.save(posting_parquet)

In [0]:
spark.sql(f"OPTIMIZE delta.`{company_dest}`")
spark.sql(f"OPTIMIZE delta.`{salary_dest}`")
spark.sql(f"OPTIMIZE delta.`{location_dest}`")
spark.sql(f"OPTIMIZE delta.`{experience_dest}`")
spark.sql(f"OPTIMIZE delta.`{monthly_dest}`")
spark.sql(f"OPTIMIZE delta.`{posting_dest}`")

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
spark.sql(f"VACUUM delta.`{company_dest}` RETAIN 168 HOURS")
spark.sql(f"VACUUM delta.`{salary_dest}` RETAIN 168 HOURS")
spark.sql(f"VACUUM delta.`{location_dest}` RETAIN 168 HOURS")
spark.sql(f"VACUUM delta.`{experience_dest}` RETAIN 168 HOURS")
spark.sql(f"VACUUM delta.`{monthly_dest}` RETAIN 168 HOURS")
spark.sql(f"VACUUM delta.`{posting_dest}` RETAIN 168 HOURS")

In [0]:
print("Gold Layer Created Successfully")